<a href="https://colab.research.google.com/github/David-kini/jeux-piere/blob/main/wav2lip-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!git clone https://huggingface.co/camenduru/Wav2Lip
!pip uninstall -y numpy librosa
!pip install numpy==1.19.5
!pip install yt_dlp ffmpeg-python librosa==0.8.0
%cd Wav2Lip

fatal: destination path 'Wav2Lip' already exists and is not an empty directory.
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: librosa 0.8.0
Uninstalling librosa-0.8.0:
  Successfully uninstalled librosa-0.8.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 115.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolv

/content/Wav2Lip/Wav2Lip


In [2]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

## Generate Talking Face Video with Wav2Lip

First, specify the paths to your input video (containing the face) and audio files. You can upload them directly to Colab or use existing paths.

In [12]:
import os

# @title Specify Video and Audio Paths

# You can upload your video and audio files directly to Colab,
# or specify paths to files already in your Colab environment.
# For example, if you uploaded 'my_face_video.mp4' and 'my_speech_audio.wav'
# to the /content/ directory, you would set the paths like this:

FACE_VIDEO_PATH = "/content/my_face_video.mp4" # @param {type:"string"}
AUDIO_PATH = "/content/my_speech_audio.wav" # @param {type:"string"}

# Check if the files exist
if not os.path.exists(FACE_VIDEO_PATH):
  print(f"Error: Face video not found at {FACE_VIDEO_PATH}")
if not os.path.exists(AUDIO_PATH):
  print(f"Error: Audio file not found at {AUDIO_PATH}")

print(f"Using face video: {FACE_VIDEO_PATH}")
print(f"Using audio file: {AUDIO_PATH}")

Error: Face video not found at /content/my_face_video.mp4
Error: Audio file not found at /content/my_speech_audio.wav
Using face video: /content/my_face_video.mp4
Using audio file: /content/my_speech_audio.wav


Now, we'll run the Wav2Lip inference script. This will combine the face video and the audio to create a new video where the face appears to be speaking the audio.

In [13]:
# Ensure you are in the Wav2Lip directory
%cd /content/Wav2Lip

# Run the inference script
# The output video will be saved in the 'results' directory within Wav2Lip
!python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face "{FACE_VIDEO_PATH}" --audio "{AUDIO_PATH}" --outfile results/output_talking_face.mp4

/content/Wav2Lip
Traceback (most recent call last):
  File "/content/Wav2Lip/inference.py", line 3, in <module>
    import scipy, cv2, os, sys, argparse, audio
  File "/content/Wav2Lip/audio.py", line 1, in <module>
    import librosa
  File "/usr/local/lib/python3.12/dist-packages/librosa/__init__.py", line 211, in <module>
    from . import core
  File "/usr/local/lib/python3.12/dist-packages/librosa/core/__init__.py", line 9, in <module>
    from .constantq import *  # pylint: disable=wildcard-import
    ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/librosa/core/constantq.py", line 1058, in <module>
    dtype=np.complex,
          ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/__init__.py", line 394, in __getattr__
    raise AttributeError(__former_attrs__[attr])
AttributeError: module 'numpy' has no attribute 'complex'.
`np.complex` was a deprecated alias for the builtin `complex`. To avoid this error in existing code, use `complex` by i

Finally, let's display the generated talking face video.

In [1]:
from IPython.display import HTML
from base64 import b64encode

output_video_path = '/content/Wav2Lip/results/output_talking_face.mp4'

if os.path.exists(output_video_path):
    mp4 = open(output_video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width="50%" height="50%" controls>
          <source src="{data_url}" type="video/mp4">
    </video>"""))
else:
    print(f"Output video not found at {output_video_path}. Please check if the inference step completed successfully.")

NameError: name 'os' is not defined